In [1]:
import json
import os
import time
import base64
import pandas as pd
import requests

api_key = 'YOUR API KEY'

In [2]:
with open('attack_qa_pairs.json', 'r') as file:
    vary_element_qa_pairs = json.load(file)
with open('attack_annotations.json', 'r') as file:
    vary_element_annotations = json.load(file)

In [6]:
counter = 0
for i, each_qa_pair in enumerate(vary_element_qa_pairs):

    task_category = each_qa_pair['type']


    if task_category == 'reasoning' and ('variance' in each_qa_pair['QA_pairs'][0]['fill_the_blank'][0] or 'deviation' in each_qa_pair['QA_pairs'][0]['fill_the_blank'][0]):
        counter+=1

print(counter)

210


In [7]:

# Function to encode the image
def encode_image(image_path):
  with open(image_path, "rb") as image_file:
    return base64.b64encode(image_file.read()).decode('utf-8')

headers = {
  "Content-Type": "application/json",
  "Authorization": f"Bearer {api_key}"
}

def get_gpt_4v_reply(image_url, final_question):
    base64_image = encode_image(image_url)
    payload = {
      "model": "gpt-4o-2024-05-13",
      "messages": [
        {
          "role": "user",
          "content": [
            {
              "type": "text",
              "text": final_question
            },
            {
              "type": "image_url",
              "image_url": {
                "url": f"data:image/png;base64,{base64_image}",
#                 'detail':'high'
              },
            }
          ]
        }
      ],
      "max_tokens": 1000
    }

    response = requests.post("https://svip.yi-zhan.top/v1/chat/completions", headers=headers, json=payload)
#     gpt_4v_total_tokens = response.json()['usage']['total_tokens']
    print(response.json())
    gpt_4v_reply = response.json()['choices'][0]['message']
    
    return gpt_4v_reply


In [ ]:

start_length = 0
total_questions = 0
gap = False
annotation_id = 0
total_question_limit = 0
break_id = -1
for _, each_annotation in enumerate(vary_element_annotations[annotation_id:]):
    total_charts = []
    if annotation_id == break_id:
        break
    # print(each_annotation)
    begin = time.time()
    image_path = each_annotation['changed_image']
    attack_type = each_annotation['attack_type']
#     vary_type = each_annotation['vary_type']
    # print(image_index)
    image_url = f'image_attack/'  + image_path
    image_type = each_annotation['type']
    index = each_annotation['id']
    for i, each_qa_pair in enumerate(vary_element_qa_pairs[start_length:]):
        each_chart = {}
        each_chart['image_url'] = image_url
        each_chart['image_type'] = image_type
        task_category = each_qa_pair['type']
        each_chart['task_category'] = task_category
        each_chart['attack_type'] = each_qa_pair['attack_type']

        if index == each_qa_pair['image_index'] and attack_type == each_qa_pair['attack_type']:
            start_length += 1
            if task_category == 'reasoning' and ('variance' in each_qa_pair['QA_pairs'][0]['fill_the_blank'][0] or 'deviation' in each_qa_pair['QA_pairs'][0]['fill_the_blank'][0]):
                continue
            total_questions += 4
            # print(total_questions)
            print(f"{index}, {task_category}")
            print(f"{attack_type}")
            for each_question_format in each_qa_pair['QA_pairs']:
                each_key = list(each_question_format.keys())[0]

                final_question = each_question_format[each_key][0]
                annotation = each_question_format[each_key][1]
                gpt_4v_reply = get_gpt_4v_reply(image_url, final_question)
                each_chart[each_key + ' question'] = final_question
                each_chart[each_key + ' annotation'] = annotation
                each_chart[each_key + ' gpt4o'] = gpt_4v_reply
            each_chart['start_length'] = start_length
            each_chart['annotation_id'] = annotation_id
            each_chart['pair_index'] = each_qa_pair['pair_index']
            total_charts.append(each_chart)
        else:
            break
        if total_questions == total_question_limit:
            gap = True
            break
    end = time.time()
    each_annotation_time = round(end - begin,2)
    annotation_id += 1
    with open(f'results/gpt4o/gpt4o_quality_2_{annotation_id}_{start_length} × 4_limit_{total_question_limit}.json', 'w') as file:
        json.dump(total_charts, file, indent=4)


6, data retrieval
original
{'id': 'chatcmpl-9XMh57LqUVpJkqBRxZZJiyrmVshkf', 'object': 'chat.completion', 'created': 1717739087, 'model': 'gpt-4o-2024-05-13', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': 'My answer is 95,096.'}, 'logprobs': None, 'finish_reason': 'stop'}], 'usage': {'prompt_tokens': 669, 'completion_tokens': 8, 'total_tokens': 677}, 'system_fingerprint': 'fp_5f4bad809a'}
{'id': 'chatcmpl-9XMhAXVu3ZIFKAvrm6rNci2OHU1WE', 'object': 'chat.completion', 'created': 1717739092, 'model': 'gpt-4o-2024-05-13', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': 'My option is 95096.'}, 'logprobs': None, 'finish_reason': 'stop'}], 'usage': {'prompt_tokens': 688, 'completion_tokens': 7, 'total_tokens': 695}, 'system_fingerprint': 'fp_5f4bad809a'}
{'id': 'chatcmpl-9XMhE5MedupfVeRuSUmixM7tuczQ2', 'object': 'chat.completion', 'created': 1717739096, 'model': 'gpt-4o-2024-05-13', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': 